In [3]:
import os 
api_key=os.getenv("SER_API")

if api_key is None:
    print("no API")
else:
    print("i find my key")

i find my key


## OpenRouter Connection

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    model="openai/gpt-4o-mini",
    temperature=0
)

response = llm.invoke("Hello")

print(response.content)

Hello! How can I assist you today?


## translate input users query  to Compilar arxiv query

In [6]:
#from langchain.prompts import PromptTemplate

from langchain_core.prompts import PromptTemplate

template = """
You are an academic retrieval query optimizer.

Your task is to transform the user's request into an optimized arXiv search query.

Rules:
1. Convert the user intent into academic search terminology.
2. Use concise research keywords.
3. Use Boolean operators (AND, OR) when useful.
4. Expand concepts using standard research terms when necessary.
5. Focus on maximizing retrieval relevance.
6. Return ONLY the final arXiv query.

User Request:
{user_input}

Optimized arXiv Query:"""

query_prompt = PromptTemplate.from_template(template)

# Function to translate query
def generate_arxiv_query(user_input):
    chain = query_prompt | llm
    response = chain.invoke({"user_input": user_input})
    return response.content.strip()

Prefix, Description, Example

`ti:`, Title, Searches for the word only in the title of the research paper.

`abs:`, Abstract, Searches the abstract of the research (the part that describes the patent or innovation).

`au:`, Author, Searches for a specific author.

`cat:`, Subject Category, Searches within a specific category (e.g., cs.AI for artificial intelligence).

`all:`, All fields, Searches in all fields (default).


In [15]:
querys = generate_arxiv_query("I want papers about using deep learning for image classification in medical imaging")
test_query= generate_arxiv_query("Jeffrey Hinton's Deep Learning Research")
print(querys)
print(test_query)

deep learning AND image classification AND medical imaging
"Jeffrey Hinton" AND ("deep learning" OR "neural networks")


## LLM Query Optimizer → arXiv Search

In [ ]:
import arxiv

def search_arxiv(query):

    client = arxiv.Client() ###

    search = arxiv.Search(
        query=query,
        max_results=2,
        sort_by=arxiv.SortCriterion.Relevance
    )
    results = []
    for result in client.results(search):
        results.append({  #Clean Retrieval Output (Good Practice)
            "paper_id": result.entry_id,
            "title": result.title,
            "year": result.published.year,
            "published": str(result.published),
            "authors": [author.name for author in result.authors],
            "summary": result.summary.replace("\n", " ").strip(),
            "url": result.pdf_url,
        })
    return results


In [ ]:
#only test
query = generate_arxiv_query(test_query)
results = search_arxiv(query)
print(query)

"Jeffrey Hinton" AND ("deep learning" OR "neural networks")


In [ ]:
#only test
q2 = "robotics AND AI"

results = search_arxiv(q2)

print(len(results))

2


In [ ]:
#only test 
for r in results:
    print("=" * 40)
    print(r["title"])

Embodied AI with Foundation Models for Mobile Service Robots: A Systematic Review
Towards Transparent Ethical AI: A Roadmap for Trustworthy Robotic Systems


In [ ]:
# only test
test_query2 = "stil papers about UAS systems"

query3 = generate_arxiv_query(test_query2)

print("QUERY:", query3)

results = search_arxiv(query3) ### this

print("RESULT COUNT:", len(results))

for r in results:
    print(r["title"])

QUERY: UAS OR "Unmanned Aerial Systems" OR "Drones" AND (applications OR technology OR "system design" OR "control systems")
RESULT COUNT: 2
Behavioral Economics for Human-in-the-loop Control Systems Design: Overconfidence and the hot hand fallacy
A Survey of Security Challenges and Solutions for UAS Traffic Management (UTM) and small Unmanned Aerial Systems (sUAS)


by llm:

- strengths
- weaknesses
- key takeaway
- critical analysis
- keywords

## Structured Schema Design